In [4]:
import numpy as np
from keras.datasets import mnist
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.utils import check_random_state
# 加载MNIST数据集
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 预处理数据
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 将数据扁平化以适应MLP模型
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat = x_test.reshape(-1, 28*28)

# 标准化转换
scaler = StandardScaler()
scaler.fit(x_train_flat)
x_train_scaled = scaler.transform(x_train_flat)
x_test_scaled = scaler.transform(x_test_flat)
# 初始化CNN模型
model_cnn = Sequential()

# 添加卷积层
model_cnn.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model_cnn.add(MaxPooling2D(pool_size=(2, 2)))
model_cnn.add(Dropout(0.25))

# 添加全连接层
model_cnn.add(Flatten())
model_cnn.add(Dense(128, activation='relu'))
model_cnn.add(Dropout(0.5))
model_cnn.add(Dense(10, activation='softmax'))

# 编译模型
model_cnn.compile(loss='categorical_crossentropy', optimizer=Adam(), metrics=['accuracy'])

# 训练模型
history_cnn = model_cnn.fit(x_train, y_train, batch_size=128, epochs=10, verbose=1, validation_data=(x_test, y_test))
# 确保 y_train 和 y_test 的形状一致
y_train_mlp = y_train.argmax(axis=1)
y_test_mlp = y_test.argmax(axis=1)

# 划分训练集和测试集
X_train_mlp, X_test_mlp, Y_train_mlp, Y_test_mlp = train_test_split(x_train_scaled, y_train_mlp, test_size=0.25, random_state=2)

# 初始化MLP模型
mlp = MLPClassifier(solver='lbfgs', hidden_layer_sizes=[200, 100], activation='relu', alpha=1, random_state=62)

# 训练模型
mlp.fit(X_train_mlp, Y_train_mlp)
# 评估CNN模型
score_cnn = model_cnn.evaluate(x_test, y_test, verbose=0)
print('CNN Test loss:', score_cnn[0])
print('CNN Test accuracy:', score_cnn[1])

# 评估MLP模型
score_mlp = mlp.score(X_test_mlp, Y_test_mlp)
print('MLP Test accuracy: {:.2f}%'.format(score_mlp * 100))

Epoch 1/10
469/469 [==============================] - 5s 10ms/step - loss: 0.3161 - accuracy: 0.9052 - val_loss: 0.0900 - val_accuracy: 0.9722
Epoch 2/10
469/469 [==============================] - 5s 10ms/step - loss: 0.1230 - accuracy: 0.9629 - val_loss: 0.0621 - val_accuracy: 0.9795
Epoch 3/10
469/469 [==============================] - 5s 11ms/step - loss: 0.0963 - accuracy: 0.9713 - val_loss: 0.0537 - val_accuracy: 0.9826
Epoch 4/10
469/469 [==============================] - 4s 9ms/step - loss: 0.0791 - accuracy: 0.9761 - val_loss: 0.0447 - val_accuracy: 0.9845
Epoch 5/10
469/469 [==============================] - 5s 11ms/step - loss: 0.0685 - accuracy: 0.9797 - val_loss: 0.0467 - val_accuracy: 0.9835
Epoch 6/10
469/469 [==============================] - 5s 10ms/step - loss: 0.0631 - accuracy: 0.9807 - val_loss: 0.0392 - val_accuracy: 0.9863
Epoch 7/10
469/469 [==============================] - 4s 9ms/step - loss: 0.0552 - accuracy: 0.9827 - val_loss: 0.0381 - val_accuracy: 0.9872
E

# 二者比较下CNN模型更适合处理图像数据